# Matched-error rank requirement

This figure complements the fixed-rank reconstruction-error plot by asking the inverse operational question: how much rank, and therefore how much normalized LoRA broadcast payload, is required to meet a target error?

For method $m$ and target error $\tau$,

$$r_{\min}^{(m)}(\tau)=\min\{r:E_r^{(m)}\leq\tau\}, \qquad \rho^{(m)}(\tau)=r_{\min}^{(m)}(\tau)/R.$$

Because the factor payload is proportional to $r(d_{\mathrm{out}}+d_{\mathrm{in}})$, $\rho$ is also the normalized broadcast payload. The RBLA band reports one standard deviation over gauge-rotation seeds.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

figure_dir = Path.cwd().resolve()
if figure_dir.name != 'matched_error_rank_requirement':
    figure_dir = figure_dir / 'rbla+' / 'figures' / 'matched_error_rank_requirement'
figure_dir.mkdir(parents=True, exist_ok=True)
source_csv = figure_dir.parent / 'broadcast_reconstruction' / 'broadcast_reconstruction_error.csv'
output_csv = figure_dir / 'matched_error_rank_requirement.csv'
output_pdf = figure_dir / 'matched_error_rank_requirement.pdf'

source = pd.read_csv(source_csv)
target_errors = np.round(np.arange(0.05, 0.801, 0.05), 2)
method_order = ['RBLA index prefix', 'SP+ (compact QR)', 'Dense truncated SVD']
missing_methods = sorted(set(method_order) - set(source['method'].unique()))
if missing_methods:
    raise ValueError(f'Missing methods in source CSV: {missing_methods}')

records = []
for (seed, method), group in source.groupby(['seed', 'method'], sort=True):
    group = group.sort_values('r')
    full_rank = int(group['R'].iloc[0])
    for target_error in target_errors:
        feasible = group.loc[group['relative_error'] <= target_error]
        if feasible.empty:
            raise ValueError(f'No feasible rank for seed={seed}, method={method}, tau={target_error}')
        selected = feasible.iloc[0]
        records.append({
            'dataset': selected['dataset'],
            'model': selected['model'],
            'experiment_type': selected['experiment_type'],
            'seed': int(seed),
            'R': full_rank,
            'target_error': float(target_error),
            'method': method,
            'required_rank': int(selected['r']),
            'normalized_required_rank': float(selected['r']) / full_rank,
            'achieved_relative_error': float(selected['relative_error']),
        })
requirements = pd.DataFrame.from_records(records)

paired = requirements.pivot(
    index=['seed', 'R', 'target_error'], columns='method',
    values='normalized_required_rank',
).reset_index().rename_axis(columns=None)
paired['normalized_rank_saving'] = (
    paired['RBLA index prefix'] - paired['SP+ (compact QR)']
)
paired['relative_rank_saving'] = (
    1.0 - paired['SP+ (compact QR)'] / paired['RBLA index prefix']
)
paired['sp_plus_dense_rank_difference'] = (
    paired['SP+ (compact QR)'] - paired['Dense truncated SVD']
)
requirements = requirements.merge(
    paired[['seed', 'R', 'target_error', 'normalized_rank_saving',
            'relative_rank_saving', 'sp_plus_dense_rank_difference']],
    on=['seed', 'R', 'target_error'], how='left',
)
summary = (
    requirements.groupby(['method', 'target_error'], as_index=False)
    .agg(mean_required_rank=('normalized_required_rank', 'mean'),
         std_required_rank=('normalized_required_rank', 'std'),
         min_required_rank=('normalized_required_rank', 'min'),
         max_required_rank=('normalized_required_rank', 'max'),
         seed_count=('seed', 'nunique'))
)
saving_summary = (
    paired.groupby('target_error', as_index=False)
    .agg(mean_normalized_rank_saving=('normalized_rank_saving', 'mean'),
         std_normalized_rank_saving=('normalized_rank_saving', 'std'),
         mean_relative_rank_saving=('relative_rank_saving', 'mean'))
)
requirements = requirements.merge(saving_summary, on='target_error', how='left')
requirements = requirements.merge(
    summary, on=['method', 'target_error'], how='left'
)
requirements.to_csv(output_csv, index=False, float_format='%.16e')

monotone = requirements.sort_values('target_error').groupby(['seed', 'method'])[
    'required_rank'
].apply(lambda values: bool(np.all(np.diff(values.to_numpy()) <= 0))).all()
checks = {
    'all_targets_are_met': bool((requirements['achieved_relative_error'] <= requirements['target_error'] + 1e-12).all()),
    'required_rank_is_valid': bool(requirements['required_rank'].between(1, requirements['R']).all()),
    'required_rank_is_monotone': bool(monotone),
    'sp_plus_matches_dense': bool((paired['sp_plus_dense_rank_difference'].abs() <= 1e-12).all()),
    'sp_plus_never_requires_more_rank': bool((paired['normalized_rank_saving'] >= -1e-12).all()),
}
if not all(checks.values()):
    raise AssertionError(checks)

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
paper_font = 'Times New Roman' if 'Times New Roman' in available_fonts else 'DejaVu Serif'
rc = {
    'font.family': paper_font,
    'font.size': 16,
    'axes.labelsize': 21,
    'xtick.labelsize': 17,
    'ytick.labelsize': 17,
    'legend.fontsize': 15,
    'axes.linewidth': 1.1,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 4.0,
    'ytick.major.size': 4.0,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
}
styles = {
    'RBLA index prefix': dict(color='#C44E52', marker='o', linestyle='-', label='RBLA prefix'),
    'SP+ (compact QR)': dict(color='#4C72B0', marker='s', linestyle='-', label='SP+ (compact QR)'),
    'Dense truncated SVD': dict(color='black', marker='^', linestyle='--', label='Dense SVD'),
}
with mpl.rc_context(rc):
    fig, ax = plt.subplots(figsize=(5.0, 5.0), dpi=600)
    rbla = summary.loc[summary['method'].eq('RBLA index prefix')].sort_values('target_error')
    x = rbla['target_error'].to_numpy(float)
    center = rbla['mean_required_rank'].to_numpy(float)
    spread = rbla['std_required_rank'].fillna(0.0).to_numpy(float)
    ax.fill_between(
        x, np.maximum(0.0, center - spread), np.minimum(1.0, center + spread),
        step='post', color=styles['RBLA index prefix']['color'],
        alpha=0.15, linewidth=0, zorder=1,
    )
    for method in method_order:
        group = summary.loc[summary['method'].eq(method)].sort_values('target_error')
        style = styles[method]
        ax.step(
            group['target_error'], group['mean_required_rank'], where='post',
            color=style['color'], linestyle=style['linestyle'], marker=style['marker'],
            linewidth=2.2, markersize=6.0, markevery=2, label=style['label'],
            markerfacecolor='white' if method == 'Dense truncated SVD' else style['color'],
            zorder=4 if method == 'Dense truncated SVD' else 3,
        )
    key_tau = 0.20
    key_rbla = float(rbla.loc[np.isclose(rbla['target_error'], key_tau), 'mean_required_rank'].iloc[0])
    sp_curve = summary.loc[summary['method'].eq('SP+ (compact QR)')].sort_values('target_error')
    key_sp = float(sp_curve.loc[np.isclose(sp_curve['target_error'], key_tau), 'mean_required_rank'].iloc[0])
    key_delta = key_rbla - key_sp
    ax.annotate('', xy=(key_tau, key_rbla), xytext=(key_tau, key_sp),
                arrowprops=dict(arrowstyle='<->', color='0.25', linewidth=1.2), zorder=6)
    ax.text(key_tau + 0.025, (key_rbla + key_sp) / 2,
            rf'$\Delta r/R={key_delta:.2f}$', fontsize=13, color='0.25',
            ha='left', va='center')
    ax.set_xlabel(r'Target relative error $\tau$')
    ax.set_ylabel(r'Minimum normalized rank $r_{\min}/R$')
    ax.set_xlim(0.045, 0.805)
    ax.set_ylim(0.10, 1.03)
    ax.set_xticks(np.arange(0.1, 0.81, 0.1))
    ax.set_yticks(np.arange(0.125, 1.001, 0.125))
    ax.set_yticklabels([f'{value:.3g}' for value in np.arange(0.125, 1.001, 0.125)])
    ax.grid(axis='both', color='0.75', linestyle=':', linewidth=0.7, alpha=0.30)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', width=1.0, length=4)
    ax.legend(loc='upper right', frameon=False, handlelength=2.1, borderaxespad=0.25)
    fig.tight_layout(pad=0.25)
    fig.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03)
    plt.show()

key_saving = saving_summary.loc[np.isclose(saving_summary['target_error'], key_tau)].iloc[0]
print(f'Source: {source_csv}')
print(f'Target errors: {target_errors.tolist()}')
print(f'Seeds per method: {requirements.seed.nunique()}')
print(f'At tau={key_tau:.2f}: SP+ requires {key_sp:.3f}R, RBLA requires {key_rbla:.3f}R')
print(f'Mean relative rank saving at tau={key_tau:.2f}: {key_saving.mean_relative_rank_saving:.1%}')
print(f'Numerical checks: {checks}')
summary